# Gold Layer - Star Schema for Power BI

Builds the business-facing star schema from `silver_market_observation`.

| Table | Type | Grain / key | Refresh |
|---|---|---|---|
| `gold_fact_market_daily` | Fact | one row per coin per day (`coin_id`, `observation_date`) | MERGE (update only if a value changed) |
| `gold_dim_coin` | Dimension | `coin_id` | MERGE (update only if an attribute changed) |
| `gold_dim_date` | Dimension | `full_date` | MERGE (insert new dates only) |
| `gold_market_summary` | Summary | one row per coin (latest observation) | MERGE (update only if a newer observation arrived) |

## What the fact table aggregates
Silver holds one row per coin per hour. Gold collapses it to one row per coin per day:
open / high / low / close / average price, closing market cap and volume, hours observed, plus
`daily_return_pct`, `moving_avg_7d_price` (7 daily rows) and `rolling_volatility_7d` (std dev of the last 7 daily returns).

## Parameters
| Widget | Behaviour |
|---|---|
| `load_type` = `full` | Rebuild every day from all of Silver. |
| `load_type` = `incremental` | Recompute days from `processing_date - 1` onward (a 14-day lookback feeds the moving windows). If the fact table is empty, it automatically runs as `full`. |
| `processing_date` | `YYYY-MM-DD`; blank = today (UTC). |

## Idempotency
Aggregation is deterministic and the MERGE updates a row only when a value changed, so re-running the same input gives 0 inserted and 0 updated.

In [0]:
# Step 1: setup, parameters, schemas and helper functions
import json, os, uuid
from datetime import date, datetime, timedelta, timezone

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType,
    LongType, IntegerType, DateType, BooleanType
)
from delta.tables import DeltaTable

dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("processing_date", "")

load_type = dbutils.widgets.get("load_type")
processing_date = dbutils.widgets.get("processing_date").strip()

if load_type not in ("full", "incremental"):
    raise ValueError("load_type must be 'full' or 'incremental'.")

if processing_date:
    date.fromisoformat(processing_date)   # fails on a bad format
elif load_type == "incremental":
    processing_date = datetime.now(timezone.utc).date().isoformat()
    print("No processing_date given; using today (UTC):", processing_date)

spark.conf.set("spark.sql.session.timeZone", "UTC")

LOOKBACK_DAYS = 14          # history needed before the first recomputed day (7-day windows + lag)
COMPLETE_DAY_HOURS = 24     # a day with fewer hourly observations is flagged as partial

silver_table_name = "workspace.default.silver_market_observation"
fact_table_name = "workspace.default.gold_fact_market_daily"
dim_coin_table_name = "workspace.default.gold_dim_coin"
dim_date_table_name = "workspace.default.gold_dim_date"
summary_table_name = "workspace.default.gold_market_summary"
log_table_name = "workspace.default.pipeline_execution_logs"

gold_run_id = str(uuid.uuid4())
gold_load_ts = datetime.now(timezone.utc).replace(tzinfo=None)

# ---------- explicit schemas (the contract for each Gold table) ----------
dim_coin_schema = StructType([
    StructField("coin_id", StringType(), False),
    StructField("symbol", StringType(), True),
    StructField("name", StringType(), True),
    StructField("market_cap_rank", LongType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

dim_date_schema = StructType([
    StructField("full_date", DateType(), False),
    StructField("day_of_month", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True),
    StructField("month_number", IntegerType(), True),
    StructField("month_name", StringType(), True),
    StructField("quarter_number", IntegerType(), True),
    StructField("year_number", IntegerType(), True),
    StructField("day_of_week_number", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("is_weekend", BooleanType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

fact_schema = StructType([
    StructField("coin_id", StringType(), False),
    StructField("observation_date", DateType(), False),
    StructField("open_price_usd", DoubleType(), True),
    StructField("high_price_usd", DoubleType(), True),
    StructField("low_price_usd", DoubleType(), True),
    StructField("close_price_usd", DoubleType(), True),
    StructField("avg_price_usd", DoubleType(), True),
    StructField("market_cap_usd", DoubleType(), True),
    StructField("total_volume_usd", DoubleType(), True),
    StructField("observation_count", LongType(), True),
    StructField("is_complete_day", BooleanType(), True),
    StructField("daily_return_pct", DoubleType(), True),
    StructField("moving_avg_7d_price", DoubleType(), True),
    StructField("rolling_volatility_7d", DoubleType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

summary_schema = StructType([
    StructField("coin_id", StringType(), False),
    StructField("latest_observation_timestamp", TimestampType(), True),
    StructField("latest_date", DateType(), True),
    StructField("price_usd", DoubleType(), True),
    StructField("price_24h_ago_usd", DoubleType(), True),
    StructField("price_change_24h_pct", DoubleType(), True),
    StructField("market_cap_usd", DoubleType(), True),
    StructField("total_volume_usd", DoubleType(), True),
    StructField("market_cap_rank_now", IntegerType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

execution_log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("processing_date", StringType(), True),
    StructField("source_folder", StringType(), True),
    StructField("start_timestamp", TimestampType(), True),
    StructField("end_timestamp", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_read", LongType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True),
])


# ---------- helper functions ----------
def enforce_schema(df, schema):
    """Select and cast columns so the DataFrame matches the explicit schema exactly."""
    missing = [f.name for f in schema.fields if f.name not in df.columns]
    if missing:
        raise ValueError(f"Columns missing for schema enforcement: {missing}")
    return df.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in schema.fields])


def table_version(table_name):
    return DeltaTable.forName(spark, table_name).history(1).select("version").first()[0]


def merge_counts(table_name, version_before):
    """Real inserted / updated counts from the Delta transaction history."""
    latest = DeltaTable.forName(spark, table_name).history(1).first()
    if latest["version"] > version_before and latest["operation"] == "MERGE":
        m = latest["operationMetrics"]
        return int(m.get("numTargetRowsInserted", 0)), int(m.get("numTargetRowsUpdated", 0))
    return 0, 0   # MERGE changed nothing


def log_gold(table_name, rows_read, rows_inserted, rows_updated, start_time, status="SUCCESS", error=None):
    """One log row per Gold table per run (key = run_id + table_name)."""
    log_df = spark.createDataFrame(
        [(
            gold_run_id, "gold", table_name, load_type, processing_date or None,
            silver_table_name, start_time, datetime.now(timezone.utc).replace(tzinfo=None),
            status, int(rows_read), int(rows_inserted), int(rows_updated), 0, error,
        )],
        schema=execution_log_schema,
    )
    (
        DeltaTable.forName(spark, log_table_name).alias("target")
        .merge(log_df.alias("source"),
               "target.run_id = source.run_id AND target.table_name = source.table_name")
        .whenNotMatchedInsertAll()
        .execute()
    )


gold_results = {}
print("Gold run ID:", gold_run_id)
print("Load type:", load_type, "| Processing date:", processing_date or "n/a")

In [0]:
# Step 2: decide how much Silver to read
silver_df = spark.table(silver_table_name)

silver_bounds = silver_df.agg(
    F.min(F.to_date("observation_timestamp")).alias("min_d"),
    F.max(F.to_date("observation_timestamp")).alias("max_d"),
).first()

if silver_bounds["min_d"] is None:
    raise ValueError("Silver is empty. Run the Silver pipeline first.")

fact_is_empty = spark.table(fact_table_name).limit(1).count() == 0

effective_load_type = load_type
if load_type == "incremental" and fact_is_empty:
    effective_load_type = "full"
    print("Gold fact table is empty, so this run is treated as a FULL load.")

if effective_load_type == "full":
    update_from = silver_bounds["min_d"]
    silver_src = silver_df
else:
    # Recompute from the day before processing_date onward. The lookback gives the
    # first recomputed day the history its moving windows and day-over-day return need.
    update_from = date.fromisoformat(processing_date) - timedelta(days=1)
    read_from = update_from - timedelta(days=LOOKBACK_DAYS)
    silver_src = silver_df.filter(F.to_date("observation_timestamp") >= F.lit(read_from))

print("Effective load type:", effective_load_type)
print("Silver date range:", silver_bounds["min_d"], "to", silver_bounds["max_d"])
print("Recomputing Gold days from:", update_from)

In [0]:
# Step 3: dim_coin (coin attributes come from the coin manifest saved by the full-load script)
step_start = datetime.now(timezone.utc).replace(tzinfo=None)

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + "/".join(notebook_path.split("/")[:-3])
except Exception:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

manifest_path = repo_root + "/data/samples/full_load_250/coin_manifest.json"

coin_rows = {}
if os.path.isfile(manifest_path):
    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)
    for c in manifest:
        if c.get("id"):
            rank = c.get("market_cap_rank")
            coin_rows[c["id"]] = (c["id"], c.get("symbol"), c.get("name"), int(rank) if rank is not None else None)
    print("Coins read from manifest:", len(coin_rows))
else:
    print("WARNING: manifest not found at", manifest_path, "- using coin ids only.")

# Every coin that appears in Silver must exist in the dimension (referential integrity)
for row in silver_df.select("coin_id").distinct().collect():
    if row[0] not in coin_rows:
        coin_rows[row[0]] = (row[0], None, row[0], None)

dim_coin_source_df = enforce_schema(
    spark.createDataFrame(
        list(coin_rows.values()),
        schema=StructType([
            StructField("coin_id", StringType(), False),
            StructField("symbol", StringType(), True),
            StructField("name", StringType(), True),
            StructField("market_cap_rank", LongType(), True),
        ]),
    ).withColumn("load_timestamp", F.lit(gold_load_ts)),
    dim_coin_schema,
)

version_before = table_version(dim_coin_table_name)
changed = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in ["symbol", "name", "market_cap_rank"])

(
    DeltaTable.forName(spark, dim_coin_table_name).alias("target")
    .merge(dim_coin_source_df.alias("source"), "target.coin_id = source.coin_id")
    .whenMatchedUpdate(
        condition=changed,
        set={"symbol": "source.symbol", "name": "source.name",
             "market_cap_rank": "source.market_cap_rank", "load_timestamp": "source.load_timestamp"},
    )
    .whenNotMatchedInsertAll()
    .execute()
)

ins, upd = merge_counts(dim_coin_table_name, version_before)
log_gold(dim_coin_table_name, len(coin_rows), ins, upd, step_start)
gold_results["gold_dim_coin"] = {"read": len(coin_rows), "inserted": ins, "updated": upd}
print("gold_dim_coin -> inserted:", ins, "| updated:", upd)

In [0]:
# Step 4: aggregate hourly Silver rows into one row per coin per day (this step shuffles the data)
daily_df = (
    silver_src
    .withColumn("observation_date", F.to_date("observation_timestamp"))
    .groupBy("coin_id", "observation_date")
    .agg(
        F.min_by("price_usd", "observation_timestamp").alias("open_price_usd"),
        F.max("price_usd").alias("high_price_usd"),
        F.min("price_usd").alias("low_price_usd"),
        F.max_by("price_usd", "observation_timestamp").alias("close_price_usd"),
        F.avg("price_usd").alias("avg_price_usd"),
        F.max_by("market_cap_usd", "observation_timestamp").alias("market_cap_usd"),
        F.max_by("total_volume_usd", "observation_timestamp").alias("total_volume_usd"),
        F.count("*").alias("observation_count"),
    )
)

by_coin = Window.partitionBy("coin_id").orderBy("observation_date")
last_7 = by_coin.rowsBetween(-6, 0)

fact_df = (
    daily_df
    .withColumn("prev_close_usd", F.lag("close_price_usd").over(by_coin))
    .withColumn(
        "daily_return_pct",
        F.when(F.col("prev_close_usd") > 0,
               (F.col("close_price_usd") / F.col("prev_close_usd") - 1) * 100),
    )
    .withColumn("moving_avg_7d_price", F.avg("close_price_usd").over(last_7))
    .withColumn("rolling_volatility_7d", F.stddev_samp("daily_return_pct").over(last_7))
    .withColumn("is_complete_day", F.col("observation_count") >= COMPLETE_DAY_HOURS)
    .filter(F.col("observation_date") >= F.lit(update_from))
    .withColumn("load_timestamp", F.lit(gold_load_ts))
)

fact_source_df = enforce_schema(fact_df, fact_schema)
print("Fact aggregation defined (nothing is computed until the MERGE runs).")

In [0]:
# Step 5: MERGE the daily fact rows into Gold
step_start = datetime.now(timezone.utc).replace(tzinfo=None)

silver_rows_read = silver_src.count()
fact_rows_built = fact_source_df.count()
print("Silver rows aggregated:", silver_rows_read, "| Daily fact rows built:", fact_rows_built)

measure_cols = [
    "open_price_usd", "high_price_usd", "low_price_usd", "close_price_usd", "avg_price_usd",
    "market_cap_usd", "total_volume_usd", "observation_count", "is_complete_day",
    "daily_return_pct", "moving_avg_7d_price", "rolling_volatility_7d",
]
changed = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in measure_cols)
update_set = {c: f"source.{c}" for c in measure_cols}
update_set["load_timestamp"] = "source.load_timestamp"

version_before = table_version(fact_table_name)
(
    DeltaTable.forName(spark, fact_table_name).alias("target")
    .merge(
        fact_source_df.alias("source"),
        "target.coin_id = source.coin_id AND target.observation_date = source.observation_date",
    )
    .whenMatchedUpdate(condition=changed, set=update_set)
    .whenNotMatchedInsertAll()
    .execute()
)

ins, upd = merge_counts(fact_table_name, version_before)
log_gold(fact_table_name, silver_rows_read, ins, upd, step_start)
gold_results["gold_fact_market_daily"] = {"read": silver_rows_read, "inserted": ins, "updated": upd}
print("gold_fact_market_daily -> inserted:", ins, "| updated:", upd)

In [0]:
# Step 6: dim_date (one row per calendar day covered by the fact table)
step_start = datetime.now(timezone.utc).replace(tzinfo=None)

fact_bounds = spark.table(fact_table_name).agg(
    F.min("observation_date").alias("d0"), F.max("observation_date").alias("d1")
).first()

dim_date_source_df = enforce_schema(
    spark.sql(
        f"SELECT explode(sequence(DATE'{fact_bounds['d0']}', DATE'{fact_bounds['d1']}', INTERVAL 1 DAY)) AS full_date"
    )
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("week_of_year", F.weekofyear("full_date"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter_number", F.quarter("full_date"))
    .withColumn("year_number", F.year("full_date"))
    .withColumn("day_of_week_number", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7))
    .withColumn("load_timestamp", F.lit(gold_load_ts)),
    dim_date_schema,
)
dim_date_rows = dim_date_source_df.count()

version_before = table_version(dim_date_table_name)
(
    DeltaTable.forName(spark, dim_date_table_name).alias("target")
    .merge(dim_date_source_df.alias("source"), "target.full_date = source.full_date")
    .whenNotMatchedInsertAll()          # calendar attributes never change, so insert-only
    .execute()
)

ins, upd = merge_counts(dim_date_table_name, version_before)
log_gold(dim_date_table_name, dim_date_rows, ins, upd, step_start)
gold_results["gold_dim_date"] = {"read": dim_date_rows, "inserted": ins, "updated": upd}
print("gold_dim_date ->", fact_bounds["d0"], "to", fact_bounds["d1"], "| inserted:", ins, "| updated:", upd)

In [0]:
# Step 7a: latest snapshot per coin (price now vs about 24 hours earlier)
silver_max_ts = silver_df.agg(F.max("observation_timestamp")).first()[0]
recent_df = silver_df.filter(F.col("observation_timestamp") >= F.lit(silver_max_ts - timedelta(days=3)))

latest_df = (
    recent_df.groupBy("coin_id").agg(
        F.max("observation_timestamp").alias("latest_observation_timestamp"),
        F.max_by("price_usd", "observation_timestamp").alias("price_usd"),
        F.max_by("market_cap_usd", "observation_timestamp").alias("market_cap_usd"),
        F.max_by("total_volume_usd", "observation_timestamp").alias("total_volume_usd"),
    )
)

prior_df = (
    recent_df
    .join(latest_df.select("coin_id", "latest_observation_timestamp"), "coin_id")
    .filter(F.col("observation_timestamp") <= F.col("latest_observation_timestamp") - F.expr("INTERVAL 24 HOURS"))
    .groupBy("coin_id")
    .agg(F.max_by("price_usd", "observation_timestamp").alias("price_24h_ago_usd"))
)

summary_df = (
    latest_df.join(prior_df, "coin_id", "left")
    .withColumn("latest_date", F.to_date("latest_observation_timestamp"))
    .withColumn(
        "price_change_24h_pct",
        F.when(F.col("price_24h_ago_usd") > 0,
               (F.col("price_usd") / F.col("price_24h_ago_usd") - 1) * 100),
    )
    .withColumn("market_cap_rank_now",
                F.row_number().over(Window.orderBy(F.col("market_cap_usd").desc_nulls_last())))
    .withColumn("load_timestamp", F.lit(gold_load_ts))
)
summary_source_df = enforce_schema(summary_df, summary_schema)
print("Summary snapshot defined.")

In [0]:
# Step 7b: MERGE the summary snapshot
step_start = datetime.now(timezone.utc).replace(tzinfo=None)
summary_rows = summary_source_df.count()

summary_cols = [
    "latest_observation_timestamp", "latest_date", "price_usd", "price_24h_ago_usd",
    "price_change_24h_pct", "market_cap_usd", "total_volume_usd", "market_cap_rank_now",
]
changed = " OR ".join(f"NOT (target.{c} <=> source.{c})" for c in summary_cols)
update_set = {c: f"source.{c}" for c in summary_cols}
update_set["load_timestamp"] = "source.load_timestamp"

version_before = table_version(summary_table_name)
(
    DeltaTable.forName(spark, summary_table_name).alias("target")
    .merge(summary_source_df.alias("source"), "target.coin_id = source.coin_id")
    .whenMatchedUpdate(condition=changed, set=update_set)
    .whenNotMatchedInsertAll()
    .execute()
)

ins, upd = merge_counts(summary_table_name, version_before)
log_gold(summary_table_name, summary_rows, ins, upd, step_start)
gold_results["gold_market_summary"] = {"read": summary_rows, "inserted": ins, "updated": upd}
print("gold_market_summary -> inserted:", ins, "| updated:", upd)

In [0]:
# Step 8: read optimisation (compacts small files; the fact table is also liquid-clustered by coin_id, observation_date)
for t in [fact_table_name, dim_coin_table_name, dim_date_table_name, summary_table_name]:
    try:
        spark.sql(f"OPTIMIZE {t}")
        print("OPTIMIZE done:", t)
    except Exception as e:
        print("OPTIMIZE skipped for", t, "->", str(e)[:200])

In [0]:
# Step 9: physical plan of the fact aggregation (use it in the report's Spark DAG analysis)
# Look for: Exchange hashpartitioning (a shuffle for the groupBy and for the window),
# and PushedFilters / PartitionFilters on the Silver scan (predicate pushdown).
fact_source_df.explain("formatted")

In [0]:
# Step 10: finish
print(json.dumps(gold_results, indent=2))
dbutils.notebook.exit(json.dumps({"run_id": gold_run_id, "status": "SUCCESS", "tables": gold_results}))